In [1]:
import torch.nn as nn
import torch
import torchvision.transforms as transforms
import torchvision
import torch.optim as optim
import numpy as np
from torch.utils.data import random_split
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
import pandas as pd
df=pd.read_csv("data_2/mitbih_train.csv")
df

,9.779411554336547852e-01,9.264705777168273926e-01,6.813725233078002930e-01,2.450980395078659058e-01,1.544117629528045654e-01,1.911764740943908691e-01,1.519607901573181152e-01,8.578431606292724609e-02,5.882352963089942932e-02,4.901960864663124084e-02,...,0.000000000000000000e+00.79,0.000000000000000000e+00.80,0.000000000000000000e+00.81,0.000000000000000000e+00.82,0.000000000000000000e+00.83,0.000000000000000000e+00.84,0.000000000000000000e+00.85,0.000000000000000000e+00.86,0.000000000000000000e+00.87,0.000000000000000000e+00.88
0,0.960114,0.863248,0.461538,0.196581,0.094017,0.125356,0.099715,0.088319,0.074074,0.082621,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.000000,0.659459,0.186486,0.070270,0.070270,0.059459,0.056757,0.043243,0.054054,0.045946,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.925414,0.665746,0.541436,0.276243,0.196133,0.077348,0.071823,0.060773,0.066298,0.058011,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.967136,1.000000,0.830986,0.586854,0.356808,0.248826,0.145540,0.089202,0.117371,0.150235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.927461,1.000000,0.626943,0.193437,0.094991,0.072539,0.043178,0.053541,0.093264,0.189983,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87548,0.807018,0.494737,0.536842,0.529825,0.491228,0.484211,0.456140,0.396491,0.284211,0.136842,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
87549,0.718333,0.605000,0.486667,0.361667,0.231667,0.120000,0.051667,0.001667,0.000000,0.013333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
87550,0.906122,0.624490,0.595918,0.575510,0.530612,0.481633,0.444898,0.387755,0.322449,0.191837,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
87551,0.858228,0.645570,0.845570,0.248101,0.167089,0.131646,0.121519,0.121519,0.118987,0.103797,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0


In [4]:
normal = df[df["0.000000000000000000e+00.88"]==0]
abnormal = df[df["0.000000000000000000e+00.88"] > 0]
X_normal = normal.drop(columns=["0.000000000000000000e+00.88"])


X_abnormal = abnormal.drop(columns=["0.000000000000000000e+00.88"])


In [5]:
normal.head()

,9.779411554336547852e-01,9.264705777168273926e-01,6.813725233078002930e-01,2.450980395078659058e-01,1.544117629528045654e-01,1.911764740943908691e-01,1.519607901573181152e-01,8.578431606292724609e-02,5.882352963089942932e-02,4.901960864663124084e-02,...,0.000000000000000000e+00.79,0.000000000000000000e+00.80,0.000000000000000000e+00.81,0.000000000000000000e+00.82,0.000000000000000000e+00.83,0.000000000000000000e+00.84,0.000000000000000000e+00.85,0.000000000000000000e+00.86,0.000000000000000000e+00.87,0.000000000000000000e+00.88
0,0.960114,0.863248,0.461538,0.196581,0.094017,0.125356,0.099715,0.088319,0.074074,0.082621,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.000000,0.659459,0.186486,0.070270,0.070270,0.059459,0.056757,0.043243,0.054054,0.045946,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.925414,0.665746,0.541436,0.276243,0.196133,0.077348,0.071823,0.060773,0.066298,0.058011,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.967136,1.000000,0.830986,0.586854,0.356808,0.248826,0.145540,0.089202,0.117371,0.150235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.927461,1.000000,0.626943,0.193437,0.094991,0.072539,0.043178,0.053541,0.093264,0.189983,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
abnormal.head()

,9.779411554336547852e-01,9.264705777168273926e-01,6.813725233078002930e-01,2.450980395078659058e-01,1.544117629528045654e-01,1.911764740943908691e-01,1.519607901573181152e-01,8.578431606292724609e-02,5.882352963089942932e-02,4.901960864663124084e-02,...,0.000000000000000000e+00.79,0.000000000000000000e+00.80,0.000000000000000000e+00.81,0.000000000000000000e+00.82,0.000000000000000000e+00.83,0.000000000000000000e+00.84,0.000000000000000000e+00.85,0.000000000000000000e+00.86,0.000000000000000000e+00.87,0.000000000000000000e+00.88
72470,1.000000,0.666667,0.100457,0.036530,0.073059,0.050228,0.018265,0.105023,0.132420,0.091324,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.0
72471,0.983696,1.000000,0.331522,0.000000,0.108696,0.163043,0.130435,0.190217,0.288043,0.222826,...,0.461957,0.483696,0.500000,0.494565,0.510870,0.51087,0.505435,0.472826,0.434783,1.0
72472,1.000000,0.911504,0.216814,0.000000,0.101770,0.199115,0.176991,0.194690,0.252212,0.238938,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.0
72473,0.090498,0.126697,0.217195,0.361991,0.461538,0.556561,0.443439,0.434389,0.452489,0.511312,...,0.122172,0.131222,0.140271,0.158371,0.176471,0.20362,0.212670,0.000000,0.000000,1.0
72474,0.961111,1.000000,0.551852,0.101852,0.040741,0.085185,0.094444,0.088889,0.085185,0.070370,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.0


In [7]:
class EGC(Dataset):
    def __init__(self,data):
        self.data = data.values
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):

        item = self.data[idx]
        return torch.tensor(item, dtype=torch.float32)

In [8]:
train = EGC(X_normal)
first_item = train[0]

print(f"Single heartbeat shape: {first_item.shape}")
val_size = int(len(train) * 0.2)
train_size = int(len(train) * 0.8)

Single heartbeat shape: torch.Size([187])


In [9]:
batch = 128
generator = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(train, [train_size, val_size], generator=generator)
trainloader = DataLoader(train_subset, batch_size=batch,
                                          shuffle=True, num_workers=0)
valloader = DataLoader(val_subset, batch_size=batch,
                       shuffle=False, num_workers=0)

In [10]:
class AutoEncodder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encode = nn.Sequential(
            nn.Linear(187,512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256,32)
        )
        self.decode = nn.Sequential(
            nn.Linear(32,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256,512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512,187),
            nn.Sigmoid()
        )
    def forward(self,x):
        en = self.encode(x)
        de = self.decode(en)
        return de

In [11]:
model = AutoEncodder().to(device)
criterion = nn.MSELoss() # Mean Squared Error is the standard for AE
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [12]:
EPOCHS = 50
for epoch in range(EPOCHS):
    train_loss = 0.0
    model.train()
    for x in trainloader:
        x = x.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output,x)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss = train_loss / len(trainloader.dataset)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x in valloader:
            ###forward
            x=x.to(device)
            output = model(x)
            loss = criterion(output,x)
            val_loss += loss.item() * x.size(0)
    val_loss = val_loss / len(valloader.dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")


Epoch 1/50 - loss: 0.0054 - val_loss: 0.0017
Epoch 2/50 - loss: 0.0016 - val_loss: 0.0012
Epoch 3/50 - loss: 0.0013 - val_loss: 0.0011
Epoch 4/50 - loss: 0.0011 - val_loss: 0.0009
Epoch 5/50 - loss: 0.0010 - val_loss: 0.0008
Epoch 6/50 - loss: 0.0010 - val_loss: 0.0008
Epoch 7/50 - loss: 0.0009 - val_loss: 0.0007
Epoch 8/50 - loss: 0.0008 - val_loss: 0.0006
Epoch 9/50 - loss: 0.0008 - val_loss: 0.0007
Epoch 10/50 - loss: 0.0008 - val_loss: 0.0006
Epoch 11/50 - loss: 0.0008 - val_loss: 0.0007
Epoch 12/50 - loss: 0.0007 - val_loss: 0.0006
Epoch 13/50 - loss: 0.0007 - val_loss: 0.0006
Epoch 14/50 - loss: 0.0007 - val_loss: 0.0006
Epoch 15/50 - loss: 0.0007 - val_loss: 0.0006
Epoch 16/50 - loss: 0.0006 - val_loss: 0.0005
Epoch 17/50 - loss: 0.0006 - val_loss: 0.0005
Epoch 18/50 - loss: 0.0006 - val_loss: 0.0005
Epoch 19/50 - loss: 0.0006 - val_loss: 0.0005
Epoch 20/50 - loss: 0.0006 - val_loss: 0.0006
Epoch 21/50 - loss: 0.0006 - val_loss: 0.0005
Epoch 22/50 - loss: 0.0006 - val_loss: 0.00

In [13]:
val_size = int(len(train) * 0.2)
train_size = int(len(train) * 0.8)
batch = 128
generator = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(train, [train_size, val_size], generator=generator)
trainloader_ab = DataLoader(train_subset, batch_size=batch,
                                          shuffle=True, num_workers=0)
valloader_ab = DataLoader(val_subset, batch_size=batch,
                       shuffle=False, num_workers=0)

In [14]:
for epoch in range(EPOCHS):
    train_loss = 0.0
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x in valloader:
            ###forward
            x=x.to(device)
            output = model(x)
            loss = criterion(output,x)
            val_loss += loss.item() * x.size(0)
    val_loss = val_loss / len(valloader.dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")


Epoch 1/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 2/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 3/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 4/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 5/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 6/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 7/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 8/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 9/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 10/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 11/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 12/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 13/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 14/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 15/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 16/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 17/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 18/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 19/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 20/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 21/50 - loss: 0.0000 - val_loss: 0.0004
Epoch 22/50 - loss: 0.0000 - val_loss: 0.00

In [15]:
model.eval()
errors = []
with torch.no_grad():
    for x in valloader:
        x = x.to(device)
        output = model(x)
        # Calculate MSE per sample
        mse = torch.mean((output - x)**2, dim=1)
        errors.extend(mse.cpu().numpy())

threshold = np.mean(errors) + (3 * np.std(errors))
print(f"Anomaly Threshold set at: {threshold:.6f}")

# 7. VALIDATION ON ABNORMAL DATA
# ------------------------------------------------------------------
abnormal_dataset = EGC(X_abnormal)
abnormal_loader = DataLoader(abnormal_dataset, batch_size=128)

abnormal_errors = []
with torch.no_grad():
    for x in abnormal_loader:
        x = x.to(device)
        output = model(x)
        mse = torch.mean((output - x)**2, dim=1)
        abnormal_errors.extend(mse.cpu().numpy())

anomalies_detected = sum(e > threshold for e in abnormal_errors)
print(f"Detected {anomalies_detected} out of {len(abnormal_errors)} abnormal beats.")

Anomaly Threshold set at: 0.001801
Detected 7217 out of 15083 abnormal beats.
